In [ ]:
import subprocess, sys, importlib

def _ensure(pkg, imp=None):
    try:
        importlib.import_module(imp or pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pkg],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )

for p in ["torch", "numpy", "gymnasium", "minigrid", "matplotlib", "scipy"]:
    _ensure(p)
_ensure("wandb")
_ensure("rllte-core", "rllte")

import torch
print(f"Python  {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}  CUDA={torch.cuda.is_available()}")

In [ ]:
# Imports
import os, json, time, random, math, zipfile
from pathlib import Path
from collections import deque, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

import gymnasium as gym
from gymnasium import spaces
from gymnasium.envs.registration import register as gym_register

import minigrid
from minigrid.core.grid import Grid
from minigrid.core.mission import MissionSpace
from minigrid.core.world_object import Ball, Box, Door, Goal, Key, Wall, Floor, Lava
from minigrid.core.constants import COLOR_NAMES
from minigrid.minigrid_env import MiniGridEnv
from minigrid.wrappers import ImgObsWrapper, RGBImgPartialObsWrapper

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False

In [ ]:
# Configuration
METHOD_NAME = "rnd_raw_dvh"
INTRINSIC_REWARD_COEFF = 0.05   # beta (external scaling of intrinsic reward)
KAPPA = 0                    # IR weight decay (0 = no decay)
SUFFIX = ""

# PPO hyper-parameters (IDENTICAL across all methods)
N_ENVS = 16
N_STEPS = 256
BATCH_SIZE = 512
N_EPOCHS = 4
LR = 2.5e-4
GAMMA = 0.999   # extrinsic discount (per RND paper)
GAE_LAMBDA = 0.95
CLIP_RANGE = 0.2
ENT_COEF = 0.01
VF_COEF = 0.5
MAX_GRAD_NORM = 0.5
ANNEAL_LR = True

# Dual value head: per RND paper — separate discounts per stream.
# Intrinsic returns are non-episodic (done signal ignored in intrinsic GAE).
GAMMA_INT = 0.99    # intrinsic discount

# Experiment
SEEDS = [1, 2, 3, 4, 5]

# Environments with step budgets (scaled by difficulty)
ENVIRONMENTS = {
    "MiniGrid-Empty-8x8-v0":                       500_000,
    "MiniGrid-LavaGapS7-v0":                     1_000_000,
    "MiniGrid-DoorKey-8x8-v0":                   2_000_000,
    "CustomMiniGrid-NoisyTVEmptyRoom-v0":         2_000_000,
    "MiniGrid-FourRooms-v0":                      5_000_000,
    "CustomMiniGrid-FourRoomsTimed-v0":            5_000_000,
    "CustomMiniGrid-BranchingCorridorsFixed-v0":   5_000_000,
    "MiniGrid-MultiRoom-N4-S5-v0":              10_000_000,
    "CustomMiniGrid-BranchingCorridors-v0":      10_000_000,
    "CustomMiniGrid-KeyChain-v0":                10_000_000,
    "CustomMiniGrid-DelayedKeySpawn-v0":         15_000_000,
    "CustomMiniGrid-TreasureIsland-v0":          15_000_000,
    "MiniGrid-KeyCorridorS6R3-v0":              30_000_000,
    "MiniGrid-ObstructedMaze-2Dlhb-v0":         50_000_000,
    "MiniGrid-ObstructedMaze-Full-v0":          100_000_000,
}

# Output
RESULTS_DIR = "./results"
USE_WANDB = True
WANDB_PROJECT = "exploration-benchmark"

# Observation rendering
TILE_SIZE = 8                  # 7x7 partial obs * 8 = 56x56 RGB images

In [ ]:
# Custom Environment Definitions
class NoisyTVEmptyRoomEnv(MiniGridEnv):
    """Empty room with a vertical curtain of noisy floor tiles.

    A column of Floor tiles divides the room — their colours randomly
    change every step.  The agent must walk *through* the noisy column
    to reach the goal on the other side.

    Tests the noisy-TV failure mode in isolation: prediction-error methods
    (ICM) waste capacity modelling the stochastic tiles while count-based
    and distillation methods are largely unaffected.
    """

    def __init__(self, size=9, **kwargs):
        self.tv_positions = []
        mission = MissionSpace(mission_func=lambda: "get to the goal")
        super().__init__(
            mission_space=mission,
            width=size,
            height=size,
            max_steps=4 * size * size,
            **kwargs,
        )

    def _gen_grid(self, width, height):
        self.grid = Grid(width, height)
        self.grid.wall_rect(0, 0, width, height)

        mid_x = width // 2
        self.tv_positions = []
        colours = [c for c in COLOR_NAMES if c != "grey"]
        for y in range(1, height - 1):
            floor = Floor(color=colours[self._rand_int(0, len(colours))])
            self.put_obj(floor, mid_x, y)
            self.tv_positions.append((mid_x, y))

        # Agent on left side
        self.place_agent(top=(1, 1), size=(mid_x - 1, height - 2))
        # Goal on right side
        self.place_obj(Goal(), top=(mid_x + 1, 1),
                       size=(width - mid_x - 2, height - 2))

    def step(self, action):
        colours = [c for c in COLOR_NAMES if c != "grey"]
        for x, y in self.tv_positions:
            obj = self.grid.get(x, y)
            if isinstance(obj, Floor):
                obj.color = colours[np.random.randint(len(colours))]
        return super().step(action)


class BranchingCorridorsEnv(MiniGridEnv):
    """Star-topology corridors radiating from a central room.

    Multiple corridors branch from a centre, only one holds the goal.
    Dead-end branches force the agent to explore systematically and
    backtrack — a challenge for methods that over-commit to a single
    trajectory.
    """

    def __init__(self, num_branches=6, branch_length=4, room_radius=1, **kwargs):
        self.num_branches = num_branches
        self.branch_length = branch_length
        self.room_radius = room_radius
        size = 2 * (branch_length + room_radius) + 3
        mission_space = MissionSpace(mission_func=lambda: "find the goal")
        super().__init__(
            mission_space=mission_space,
            width=size,
            height=size,
            max_steps=16 * size * size,
            **kwargs,
        )

    def _gen_grid(self, width, height):
        self.grid = Grid(width, height)
        # Fill with walls
        for x in range(width):
            for y in range(height):
                self.grid.set(x, y, Wall())

        cx, cy = width // 2, height // 2
        r = self.room_radius

        # Central room
        for dx in range(-r, r + 1):
            for dy in range(-r, r + 1):
                self.grid.set(cx + dx, cy + dy, None)

        # Carve corridors
        ends = []
        for i in range(self.num_branches):
            angle = 2 * math.pi * i / self.num_branches
            cos_a, sin_a = math.cos(angle), math.sin(angle)
            for step in range(1, self.branch_length + r + 1):
                x = cx + int(round(cos_a * step))
                y = cy + int(round(sin_a * step))
                if 1 <= x < width - 1 and 1 <= y < height - 1:
                    self.grid.set(x, y, None)
            ex = cx + int(round(cos_a * (self.branch_length + r)))
            ey = cy + int(round(sin_a * (self.branch_length + r)))
            if 1 <= ex < width - 1 and 1 <= ey < height - 1:
                ends.append((ex, ey))

        # Goal at one random branch end
        if ends:
            gx, gy = ends[self._rand_int(0, len(ends))]
            self.put_obj(Goal(), gx, gy)

        # Agent at centre
        self.agent_pos = np.array([cx, cy])
        self.agent_dir = self._rand_int(0, 4)

        # Outer walls (overwrite)
        self.grid.wall_rect(0, 0, width, height)


class BranchingCorridorsFixedEnv(MiniGridEnv):
    """Star-topology corridors with the goal at a fixed branch.

    Identical to BranchingCorridorsEnv except the goal is always placed
    at the first branch end, making the layout fully deterministic.
    Tests whether agents can memorise a fixed path vs requiring
    systematic exploration every episode.
    """

    def __init__(self, num_branches=6, branch_length=4, room_radius=1, **kwargs):
        self.num_branches = num_branches
        self.branch_length = branch_length
        self.room_radius = room_radius
        size = 2 * (branch_length + room_radius) + 3
        mission_space = MissionSpace(mission_func=lambda: "find the goal")
        super().__init__(
            mission_space=mission_space,
            width=size,
            height=size,
            max_steps=16 * size * size,
            **kwargs,
        )

    def _gen_grid(self, width, height):
        self.grid = Grid(width, height)
        for x in range(width):
            for y in range(height):
                self.grid.set(x, y, Wall())

        cx, cy = width // 2, height // 2
        r = self.room_radius

        for dx in range(-r, r + 1):
            for dy in range(-r, r + 1):
                self.grid.set(cx + dx, cy + dy, None)

        ends = []
        for i in range(self.num_branches):
            angle = 2 * math.pi * i / self.num_branches
            cos_a, sin_a = math.cos(angle), math.sin(angle)
            for step in range(1, self.branch_length + r + 1):
                x = cx + int(round(cos_a * step))
                y = cy + int(round(sin_a * step))
                if 1 <= x < width - 1 and 1 <= y < height - 1:
                    self.grid.set(x, y, None)
            ex = cx + int(round(cos_a * (self.branch_length + r)))
            ey = cy + int(round(sin_a * (self.branch_length + r)))
            if 1 <= ex < width - 1 and 1 <= ey < height - 1:
                ends.append((ex, ey))

        # Goal at FIXED branch (always the first one)
        if ends:
            gx, gy = ends[0]
            self.put_obj(Goal(), gx, gy)

        self.agent_pos = np.array([cx, cy])
        self.agent_dir = self._rand_int(0, 4)

        self.grid.wall_rect(0, 0, width, height)


class DelayedKeySpawnEnv(MiniGridEnv):
    """Two-room env where the key only appears after a trigger tile is visited.

    Causal chain the agent must discover:
      1. step on green trigger  ->  yellow key spawns
      2. pick up key            ->  unlock yellow door
      3. enter room 2           ->  reach goal

    Tests multi-step causal reasoning in exploration.
    """

    def __init__(self, size=10, **kwargs):
        self.key_spawned = False
        self.trigger_pos = None
        self.key_spawn_pos = None
        mission = MissionSpace(
            mission_func=lambda: "activate the trigger, get the key, reach the goal"
        )
        super().__init__(
            mission_space=mission,
            width=size,
            height=size,
            max_steps=10 * size * size,
            **kwargs,
        )

    def _gen_grid(self, width, height):
        self.grid = Grid(width, height)
        self.grid.wall_rect(0, 0, width, height)

        # Dividing wall
        wx = width // 2
        for y in range(height):
            self.grid.set(wx, y, Wall())

        # Locked door
        dy = self._rand_int(2, height - 2)
        self.put_obj(Door("yellow", is_locked=True), wx, dy)

        # Goal in right room
        self.put_obj(Goal(), self._rand_int(wx + 2, width - 1),
                     self._rand_int(1, height - 1))

        # Trigger (green floor) in left room
        self.trigger_pos = (
            self._rand_int(1, wx - 1),
            self._rand_int(1, height - 1),
        )
        self.put_obj(Floor("green"), *self.trigger_pos)

        # Key spawn location (grey floor marker) in left room
        self.key_spawn_pos = (
            self._rand_int(1, wx - 1),
            self._rand_int(1, height - 1),
        )
        while self.key_spawn_pos == self.trigger_pos:
            self.key_spawn_pos = (
                self._rand_int(1, wx - 1),
                self._rand_int(1, height - 1),
            )
        self.put_obj(Floor("grey"), *self.key_spawn_pos)

        # Agent in left room
        self.place_agent(size=(wx, height))
        self.key_spawned = False

    def step(self, action):
        if not self.key_spawned:
            ax, ay = int(self.agent_pos[0]), int(self.agent_pos[1])
            if (ax, ay) == self.trigger_pos:
                self.grid.set(*self.key_spawn_pos, Key("yellow"))
                self.grid.set(*self.trigger_pos, Floor("red"))
                self.key_spawned = True
        return super().step(action)


class FourRoomsTimedEnv(MiniGridEnv):
    """FourRooms where the goal respawns instead of ending the episode.

    Identical grid to MiniGrid-FourRooms-v0 (19x19, four connected rooms
    with randomised door positions).  When the agent reaches the goal it
    receives reward and the goal respawns at a new random position.
    The episode ends only at max_steps.

    Tests reward-collection efficiency: the agent must learn the room
    layout and navigate between goals as fast as possible.
    """

    def __init__(self, size=19, max_steps=500, **kwargs):
        mission = MissionSpace(
            mission_func=lambda: "collect as many goals as possible")
        super().__init__(
            mission_space=mission,
            width=size,
            height=size,
            max_steps=max_steps,
            **kwargs,
        )

    def _gen_grid(self, width, height):
        # Exact replica of MiniGrid FourRooms grid generation
        self.grid = Grid(width, height)
        self.grid.horz_wall(0, 0)
        self.grid.horz_wall(0, height - 1)
        self.grid.vert_wall(0, 0)
        self.grid.vert_wall(width - 1, 0)

        room_w = width // 2
        room_h = height // 2

        for j in range(2):
            for i in range(2):
                xL = i * room_w
                yT = j * room_h
                xR = xL + room_w
                yB = yT + room_h
                if i + 1 < 2:
                    self.grid.vert_wall(xR, yT, room_h)
                    pos = (xR, self._rand_int(yT + 1, yB))
                    self.grid.set(*pos, None)
                if j + 1 < 2:
                    self.grid.horz_wall(xL, yB, room_w)
                    pos = (self._rand_int(xL + 1, xR), yB)
                    self.grid.set(*pos, None)

        self.place_agent()
        self.place_obj(Goal())

    def step(self, action):
        fwd_pos = self.front_pos.copy()
        fwd_cell = self.grid.get(*fwd_pos)

        obs, reward, terminated, truncated, info = super().step(action)

        # If we hit a goal, don't terminate — respawn it
        if fwd_cell is not None and fwd_cell.type == "goal" and terminated:
            terminated = False
            self.grid.set(fwd_pos[0], fwd_pos[1], None)
            self.place_obj(Goal())

        return obs, reward, terminated, truncated, info


class KeyChainEnv(MiniGridEnv):
    """Fixed-layout dungeon with a sequential key-door chain.

    Three rooms in a row connected by locked doors:
        Room 1 -> [yellow door] -> Room 2 -> [green door] -> Room 3 -> Goal

    The agent must pick up the yellow key in room 1, unlock the yellow
    door, pick up the green key in room 2, unlock the green door, then
    reach the goal in room 3.  Intermediate rewards for key pickups.

    Fixed layout (Atari-like): the map is identical every episode,
    so the challenge is learning the causal chain, not mapping.
    """

    def __init__(self, **kwargs):
        mission = MissionSpace(
            mission_func=lambda: "collect keys, unlock doors, reach the goal")
        super().__init__(
            mission_space=mission,
            width=19,
            height=9,
            max_steps=300,
            **kwargs,
        )

    def _gen_grid(self, width, height):
        self.grid = Grid(width, height)
        self.grid.wall_rect(0, 0, width, height)

        # Internal walls dividing into 3 rooms
        for y in range(height):
            self.grid.set(6, y, Wall())
            self.grid.set(12, y, Wall())

        # Locked doors
        self.put_obj(Door("yellow", is_locked=True), 6, 4)
        self.put_obj(Door("green", is_locked=True), 12, 4)

        # Keys at fixed positions
        self.put_obj(Key("yellow"), 3, 6)
        self.put_obj(Key("green"), 9, 2)

        # Goal in room 3
        self.put_obj(Goal(), 16, 4)

        # Agent in room 1 (fixed position)
        self.agent_pos = np.array([2, 4])
        self.agent_dir = 0  # facing right

    def step(self, action):
        carrying_before = self.carrying
        obs, reward, terminated, truncated, info = super().step(action)

        # Intermediate reward for picking up a key
        if self.carrying is not None and carrying_before is None:
            if isinstance(self.carrying, Key):
                reward += 0.2

        return obs, reward, terminated, truncated, info


class TreasureIslandEnv(MiniGridEnv):
    """Fixed-layout four-room island with treasures and locked doors.

    Four quadrant rooms arranged in a 2x2 grid:
        Room A (top-left):     agent start
        Room B (top-right):    locked (yellow door from A)
        Room C (bottom-left):  open passage from A, yellow key here
        Room D (bottom-right): locked (blue door from C)

    Exploration sequence:  A -> C (get yellow key) -> A -> B (get blue key)
                          -> A -> C -> D

    Green floor tiles in each room act as treasures (+0.25 reward on
    first visit).  Key pickups give +0.1.  Episode ends at max_steps
    only — no terminal goal.

    Fixed layout (Atari-like): tests breadth exploration and multi-step
    planning in a known environment.
    """

    def __init__(self, **kwargs):
        self.collected = set()
        self.treasure_positions = set()
        mission = MissionSpace(
            mission_func=lambda: "collect all four treasures")
        super().__init__(
            mission_space=mission,
            width=11,
            height=11,
            max_steps=300,
            **kwargs,
        )

    def _gen_grid(self, width, height):
        self.grid = Grid(width, height)
        self.grid.wall_rect(0, 0, width, height)
        self.collected = set()

        # Cross-shaped internal walls
        for x in range(1, width - 1):
            self.grid.set(x, 5, Wall())
        for y in range(1, height - 1):
            self.grid.set(5, y, Wall())

        # Connections
        self.grid.set(2, 5, None)                            # A<->C open passage
        self.put_obj(Door("yellow", is_locked=True), 5, 2)  # A<->B yellow door
        self.put_obj(Door("blue", is_locked=True), 5, 8)  # C<->D blue door

        # Keys
        self.put_obj(Key("yellow"), 2, 8)   # yellow key in room C
        self.put_obj(Key("blue"), 8, 2)   # blue key in room B

        # Treasures (green floor tiles)
        self.treasure_positions = {(3, 3), (7, 3), (3, 7), (7, 7)}
        for pos in self.treasure_positions:
            self.put_obj(Floor("green"), *pos)

        # Agent in room A (fixed)
        self.agent_pos = np.array([2, 2])
        self.agent_dir = 0  # facing right

    def step(self, action):
        carrying_before = self.carrying
        obs, reward, terminated, truncated, info = super().step(action)

        # Reward for key pickup
        if self.carrying is not None and carrying_before is None:
            if isinstance(self.carrying, Key):
                reward += 0.1

        # Reward for stepping on a treasure
        pos = (int(self.agent_pos[0]), int(self.agent_pos[1]))
        if pos in self.treasure_positions and pos not in self.collected:
            reward += 0.25
            self.collected.add(pos)
            self.grid.set(*pos, None)

        return obs, reward, terminated, truncated, info


# Register custom environments
_CUSTOM_REGISTERED = False

def register_custom_envs():
    global _CUSTOM_REGISTERED
    if _CUSTOM_REGISTERED:
        return
    gym_register(id="CustomMiniGrid-NoisyTVEmptyRoom-v0",
                 entry_point="__main__:NoisyTVEmptyRoomEnv")
    gym_register(id="CustomMiniGrid-BranchingCorridors-v0",
                 entry_point="__main__:BranchingCorridorsEnv")
    gym_register(id="CustomMiniGrid-BranchingCorridorsFixed-v0",
                 entry_point="__main__:BranchingCorridorsFixedEnv")
    gym_register(id="CustomMiniGrid-DelayedKeySpawn-v0",
                 entry_point="__main__:DelayedKeySpawnEnv")
    gym_register(id="CustomMiniGrid-FourRoomsTimed-v0",
                 entry_point="__main__:FourRoomsTimedEnv")
    gym_register(id="CustomMiniGrid-KeyChain-v0",
                 entry_point="__main__:KeyChainEnv")
    gym_register(id="CustomMiniGrid-TreasureIsland-v0",
                 entry_point="__main__:TreasureIslandEnv")
    _CUSTOM_REGISTERED = True
    print("Custom environments registered.")

register_custom_envs()

In [ ]:
# Environment Factory

class TransposeObsWrapper(gym.ObservationWrapper):
    """Transpose observations from HWC (gymnasium default) to CHW (PyTorch / RLeXplore)."""
    def __init__(self, env):
        super().__init__(env)
        h, w, c = env.observation_space.shape
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=(c, h, w), dtype=np.uint8,
        )
    def observation(self, obs):
        return np.transpose(obs, (2, 0, 1))


def make_env(env_id, seed, idx):
    """Return a thunk that creates one wrapped MiniGrid environment."""
    def thunk():
        env = gym.make(env_id, render_mode=None)
        env = RGBImgPartialObsWrapper(env, tile_size=TILE_SIZE)
        env = ImgObsWrapper(env)
        env = TransposeObsWrapper(env)      # HWC -> CHW
        env = gym.wrappers.RecordEpisodeStatistics(env)
        env.reset(seed=seed + idx)
        env.action_space.seed(seed + idx)
        env.observation_space.seed(seed + idx)
        return env
    return thunk


def make_envs(env_id, seed, n_envs=None):
    """Create a SyncVectorEnv with n_envs parallel copies."""
    n = n_envs or N_ENVS
    return gym.vector.SyncVectorEnv([make_env(env_id, seed, i) for i in range(n)])

In [ ]:
# Neural Network (Actor-Critic with CNN — Dual Value Head)
# Identical to the single-head version except critic is split into
# critic_ext (extrinsic) and critic_int (intrinsic).  Shared encoder.

def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    """Orthogonal weight init (standard for PPO)."""
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, bias_const)
    return layer


class ActorCritic(nn.Module):
    """CNN actor-critic with dual value heads for 3x56x56 MiniGrid RGB partial
    observations (CHW).

    Architecture mirrors Nature-DQN / CleanRL PPO:
        Conv(3->32, 8, 4) -> Conv(32->64, 4, 2) -> Conv(64->64, 3, 1)
        -> Flatten(576) -> Linear(512) -> actor / critic_ext / critic_int heads
    """

    def __init__(self, obs_shape, n_actions):
        super().__init__()
        c = obs_shape[0]  # channels first: (C, H, W) after TransposeObsWrapper

        self.encoder = nn.Sequential(
            layer_init(nn.Conv2d(c, 32, 8, stride=4)),
            nn.ReLU(),
            layer_init(nn.Conv2d(32, 64, 4, stride=2)),
            nn.ReLU(),
            layer_init(nn.Conv2d(64, 64, 3, stride=1)),
            nn.ReLU(),
            nn.Flatten(),
            layer_init(nn.Linear(64 * 3 * 3, 512)),
            nn.ReLU(),
        )
        self.actor = layer_init(nn.Linear(512, n_actions), std=0.01)
        self.critic_ext = layer_init(nn.Linear(512, 1), std=1.0)
        self.critic_int = layer_init(nn.Linear(512, 1), std=1.0)

    def _encode(self, x):
        # x : (B, C, H, W) uint8 → float32 [0,1]  (already CHW via TransposeObsWrapper)
        return self.encoder(x.float() / 255.0)

    def get_value(self, x):
        h = self._encode(x)
        return self.critic_ext(h).squeeze(-1), self.critic_int(h).squeeze(-1)

    def get_action_and_value(self, x, action=None):
        h = self._encode(x)
        logits = self.actor(h)
        dist = Categorical(logits=logits)
        if action is None:
            action = dist.sample()
        return (action, dist.log_prob(action), dist.entropy(),
                self.critic_ext(h).squeeze(-1), self.critic_int(h).squeeze(-1))

In [ ]:
# RND 2-head

class _RunningMeanStd:
    """Welford's online running mean / variance."""
    def __init__(self):
        self.mean = 0.0
        self.var = 1.0
        self.count = 1e-4
    def update(self, x):
        batch_mean = np.mean(x)
        batch_var = np.var(x)
        batch_count = max(np.size(x), 1)
        delta = batch_mean - self.mean
        tot = self.count + batch_count
        self.mean  += delta * batch_count / tot
        m_a = self.var  * self.count
        m_b = batch_var * batch_count
        self.var = (m_a + m_b + delta**2 * self.count * batch_count / tot) / tot
        self.count = tot


class _RNDEncoder(nn.Module):
    """Mnih-style CNN encoder for RND target/predictor networks (56x56 input)."""
    def __init__(self, in_c, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, 32, 8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 3 * 3, latent_dim),
        )
        # Orthogonal init
        for m in self.net:
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.orthogonal_(m.weight, np.sqrt(2))
                nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)


class RNDRawModule:
    def __init__(self, envs, device, latent_dim=128, lr=1e-3):
        self.device = device
        n_envs = envs.num_envs
        c = envs.single_observation_space.shape[0]  # CHW format

        self.target = _RNDEncoder(c, latent_dim).to(device)
        self.predictor = _RNDEncoder(c, latent_dim).to(device)
        for p in self.target.parameters():
            p.requires_grad = False

        self.opt = optim.Adam(self.predictor.parameters(), lr=lr)
        self.rms = _RunningMeanStd()

    @torch.no_grad()
    def _pred_error(self, obs_chw):
        tgt = self.target(obs_chw)
        pred = self.predictor(obs_chw)
        return ((tgt - pred) ** 2).mean(dim=-1)

    def compute(self, obs, nobs, actions, dones):
        n_steps, n_envs = obs.shape[:2]
        no = nobs.reshape(-1, *nobs.shape[2:]).float() / 255.0
        with torch.no_grad():
            ir = self._pred_error(no).reshape(n_steps, n_envs)
        return ir.detach()  # no normalisation — raw IR decays naturally

    def update(self, obs, nobs, actions):
        flat = nobs.reshape(-1, *nobs.shape[2:]).float() / 255.0
        n = flat.shape[0]
        idx = torch.randint(0, n, (min(n, 256),))
        batch = flat[idx].to(self.device)
        tgt = self.target(batch).detach()
        pred = self.predictor(batch)
        loss = ((tgt - pred) ** 2).mean()
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()


def create_intrinsic_reward_module(envs, device):
    return RNDRawModule(envs, device, latent_dim=128, lr=1e-3)

def watch_intrinsic_module(irs, obs, actions, rewards, terminateds, truncateds, next_obs):
    pass  # RND only needs next_obs at compute time, no per-step tracking

def compute_intrinsic_rewards(irs, obs, nobs, actions, terminateds, truncateds, rewards):
    dones = terminateds + truncateds
    return irs.compute(obs, nobs, actions, dones)

def update_intrinsic_module(irs, obs, nobs, actions, terminateds, truncateds, rewards):
    irs.update(obs, nobs, actions)

In [ ]:
# Training Loop  (PPO + optional intrinsic reward — Dual Value Head)
# Identical to the single-head training loop except:

def train(env_id, total_timesteps, seed, device):
    """Run a single training experiment: method × environment × seed."""
    _full_name = f"{METHOD_NAME}{SUFFIX}"
    run_name = f"{_full_name}__{env_id}__{seed}"
    result_dir = Path(RESULTS_DIR) / run_name

    # Skip if already completed
    if (result_dir / "summary.json").exists():
        print(f"  [SKIP] {run_name}")
        with open(result_dir / "summary.json") as f:
            return json.load(f)

    # Seeding
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

    # Environments
    envs = make_envs(env_id, seed)
    obs_shape = envs.single_observation_space.shape   # (3, 56, 56) after CHW transpose
    n_actions = envs.single_action_space.n

    # Agent
    agent = ActorCritic(obs_shape, n_actions).to(device)
    optimizer = optim.Adam(agent.parameters(), lr=LR, eps=1e-5)

    # Intrinsic-reward module (method-specific)
    irs = create_intrinsic_reward_module(envs, device)

    # Rollout buffers
    obs_buf = torch.zeros((N_STEPS, N_ENVS) + obs_shape,
                               dtype=torch.uint8, device=device)
    nobs_buf = torch.zeros_like(obs_buf)
    act_buf = torch.zeros((N_STEPS, N_ENVS),
                               dtype=torch.long, device=device)
    rew_buf = torch.zeros((N_STEPS, N_ENVS), device=device)
    term_buf = torch.zeros((N_STEPS, N_ENVS), device=device)
    trunc_buf = torch.zeros((N_STEPS, N_ENVS), device=device)
    done_buf = torch.zeros((N_STEPS, N_ENVS), device=device)
    logp_buf = torch.zeros((N_STEPS, N_ENVS), device=device)
    val_ext_buf = torch.zeros((N_STEPS, N_ENVS), device=device)
    val_int_buf = torch.zeros((N_STEPS, N_ENVS), device=device)

    # Logging arrays
    ep_returns, ep_lengths, ep_steps = [], [], []
    ir_means = []                         # mean intrinsic reward per update

    # Initialise
    obs_np, _ = envs.reset(seed=seed)
    obs = torch.from_numpy(obs_np).to(device)
    done = torch.zeros(N_ENVS, device=device)

    num_updates = total_timesteps // (N_ENVS * N_STEPS)
    global_step = 0
    start_time = time.time()
    log_interval = max(1, num_updates // 50)

    if USE_WANDB and WANDB_AVAILABLE:
        wandb.init(project=WANDB_PROJECT, name=run_name,
                   config=dict(method=_full_name, env_id=env_id, seed=seed,
                               total_timesteps=total_timesteps,
                               intrinsic_reward_coeff=INTRINSIC_REWARD_COEFF,
                               gamma_int=GAMMA_INT, value_heads=2),
                   reinit=True)

    for update in range(1, num_updates + 1):

        # LR annealing
        if ANNEAL_LR:
            frac = 1.0 - (update - 1) / num_updates
            optimizer.param_groups[0]["lr"] = LR * frac

        # Collect rollout
        for step in range(N_STEPS):
            global_step += N_ENVS
            obs_buf[step] = obs
            done_buf[step] = done

            with torch.no_grad():
                action, logprob, _, value_ext, value_int = \
                    agent.get_action_and_value(obs)
            act_buf[step] = action
            logp_buf[step] = logprob
            val_ext_buf[step] = value_ext
            val_int_buf[step] = value_int

            obs_np, reward, terminated, truncated, info = \
                envs.step(action.cpu().numpy())
            term_buf[step] = torch.from_numpy(
                np.asarray(terminated, dtype=np.float32)).to(device)
            trunc_buf[step] = torch.from_numpy(
                np.asarray(truncated, dtype=np.float32)).to(device)
            done = term_buf[step] + trunc_buf[step]
            rew_buf[step] = torch.from_numpy(
                np.asarray(reward, dtype=np.float32)).to(device)
            obs = torch.from_numpy(obs_np).to(device)
            nobs_buf[step] = obs

            # Let the intrinsic-reward module observe each step
            if irs is not None:
                watch_intrinsic_module(
                    irs, obs_buf[step], action, rew_buf[step],
                    term_buf[step], trunc_buf[step], obs)

            # gymnasium >=1.0 puts episode stats in info["episode"]
            # with a boolean mask info["_episode"]
            if "_episode" in info:
                for i, flag in enumerate(info["_episode"]):
                    if flag:
                        _r = float(info["episode"]["r"][i])
                        _l = int(info["episode"]["l"][i])
                        ep_returns.append(_r)
                        ep_lengths.append(_l)
                        ep_steps.append(global_step)
                        if USE_WANDB and WANDB_AVAILABLE:
                            wandb.log({
                                "charts/episode_reward": _r,
                                "charts/episode_length": _l,
                                "global_step": global_step,
                            })
            # gymnasium <1.0 uses info["final_info"]
            elif "final_info" in info:
                for item in info["final_info"]:
                    if item is not None and "episode" in item:
                        _r = float(item["episode"]["r"])
                        _l = int(item["episode"]["l"])
                        ep_returns.append(_r)
                        ep_lengths.append(_l)
                        ep_steps.append(global_step)
                        if USE_WANDB and WANDB_AVAILABLE:
                            wandb.log({
                                "charts/episode_reward": _r,
                                "charts/episode_length": _l,
                                "global_step": global_step,
                            })

        # Intrinsic rewards
        if irs is not None:
            intrinsic = compute_intrinsic_rewards(
                irs, obs_buf, nobs_buf, act_buf, term_buf, trunc_buf, rew_buf)
            ir_means.append(float(intrinsic.mean()))
        else:
            intrinsic = None

        # GAE (separate for extrinsic and intrinsic)
        with torch.no_grad():
            next_value_ext, next_value_int = agent.get_value(obs)

            # Extrinsic GAE — trained on raw environment rewards only
            advantages_ext = torch.zeros_like(rew_buf)
            lastgaelam = 0.0
            for t in reversed(range(N_STEPS)):
                if t == N_STEPS - 1:
                    nonterminal = 1.0 - done
                    next_val = next_value_ext
                else:
                    nonterminal = 1.0 - done_buf[t + 1]
                    next_val = val_ext_buf[t + 1]
                delta = rew_buf[t] + GAMMA * next_val * nonterminal - val_ext_buf[t]
                advantages_ext[t] = lastgaelam = \
                    delta + GAMMA * GAE_LAMBDA * nonterminal * lastgaelam
            returns_ext = advantages_ext + val_ext_buf

            # Intrinsic GAE — non-episodic per RND paper: done signal is ignored
            # so novelty value accumulates across episode boundaries.
            if intrinsic is not None:
                advantages_int = torch.zeros_like(intrinsic)
                lastgaelam = 0.0
                for t in reversed(range(N_STEPS)):
                    if t == N_STEPS - 1:
                        next_val = next_value_int
                    else:
                        next_val = val_int_buf[t + 1]
                    # No nonterminal mask — intrinsic bootstrap never cut at episode end
                    delta = intrinsic[t] + GAMMA_INT * next_val - val_int_buf[t]
                    advantages_int[t] = lastgaelam = \
                        delta + GAMMA_INT * GAE_LAMBDA * lastgaelam
                returns_int = advantages_int + val_int_buf
            else:
                advantages_int = torch.zeros_like(advantages_ext)
                returns_int = torch.zeros_like(returns_ext)

        # Flatten
        bs = N_ENVS * N_STEPS
        b_obs = obs_buf.reshape((-1,) + obs_shape)
        b_logp = logp_buf.reshape(-1)
        b_act = act_buf.reshape(-1)
        # Combined advantage for policy gradient
        b_adv = (advantages_ext +
                     INTRINSIC_REWARD_COEFF * advantages_int).reshape(-1)
        b_ret_ext = returns_ext.reshape(-1)
        b_ret_int = returns_int.reshape(-1)

        # PPO update
        pg_losses, v_losses_ext, v_losses_int, ent_vals, clipfracs, approx_kls = \
            [], [], [], [], [], []
        for _epoch in range(N_EPOCHS):
            idx = torch.randperm(bs, device=device)
            for start in range(0, bs, BATCH_SIZE):
                mb = idx[start : start + BATCH_SIZE]

                _, newlogp, entropy, new_val_ext, new_val_int = \
                    agent.get_action_and_value(b_obs[mb], b_act[mb])
                logratio = newlogp - b_logp[mb]
                ratio = logratio.exp()

                with torch.no_grad():
                    approx_kl = ((ratio - 1) - logratio).mean().item()
                    approx_kls.append(approx_kl)
                    clipfracs.append(
                        ((ratio - 1).abs() > CLIP_RANGE).float().mean().item())

                mb_adv = b_adv[mb]
                mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)

                pg1 = -mb_adv * ratio
                pg2 = -mb_adv * ratio.clamp(1 - CLIP_RANGE, 1 + CLIP_RANGE)
                pg_loss = torch.max(pg1, pg2).mean()
                v_loss_ext = 0.5 * ((new_val_ext - b_ret_ext[mb]) ** 2).mean()
                v_loss_int = 0.5 * ((new_val_int - b_ret_int[mb]) ** 2).mean()
                v_loss = v_loss_ext + v_loss_int
                ent_loss = entropy.mean()

                loss = pg_loss - ENT_COEF * ent_loss + VF_COEF * v_loss

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(agent.parameters(), MAX_GRAD_NORM)
                optimizer.step()

                pg_losses.append(pg_loss.item())
                v_losses_ext.append(v_loss_ext.item())
                v_losses_int.append(v_loss_int.item())
                ent_vals.append(ent_loss.item())

        # Update intrinsic module
        if irs is not None:
            update_intrinsic_module(irs, obs_buf, nobs_buf, act_buf,
                                    term_buf, trunc_buf, rew_buf)

        # Logging
        if update % log_interval == 0 or update == num_updates:
            elapsed = time.time() - start_time
            sps = global_step / max(elapsed, 1e-6)
            recent = ep_returns[-100:] if ep_returns else [0.0]
            avg_ret = np.mean(recent)
            pct = 100.0 * update / num_updates
            ir_str = f" | IR {ir_means[-1]:.4f}" if ir_means else ""
            print(f"  [{pct:5.1f}%] step {global_step:>10,} | "
                  f"ret {avg_ret:>8.3f} | SPS {sps:>6.0f}{ir_str}")
            if USE_WANDB and WANDB_AVAILABLE:
                log_dict = {"global_step": global_step,
                           "charts/episodic_return": avg_ret,
                           "charts/SPS": sps,
                           "losses/policy": np.mean(pg_losses),
                           "losses/value": np.mean(v_losses_ext) + np.mean(v_losses_int),
                           "losses/value_ext": np.mean(v_losses_ext),
                           "losses/value_int": np.mean(v_losses_int),
                           "losses/entropy": np.mean(ent_vals),
                           "losses/approx_kl": np.mean(approx_kls),
                           "losses/clipfrac": np.mean(clipfracs)}
                if ir_means:
                    log_dict["charts/intrinsic_reward"] = ir_means[-1]
                wandb.log(log_dict)

    # Save results
    result_dir.mkdir(parents=True, exist_ok=True)

    config = dict(
        method=_full_name, env_id=env_id, seed=seed,
        total_timesteps=total_timesteps,
        n_envs=N_ENVS, n_steps=N_STEPS, batch_size=BATCH_SIZE,
        n_epochs=N_EPOCHS, lr=LR, gamma=GAMMA, gae_lambda=GAE_LAMBDA,
        clip_range=CLIP_RANGE, ent_coef=ENT_COEF, vf_coef=VF_COEF,
        max_grad_norm=MAX_GRAD_NORM, intrinsic_reward_coeff=INTRINSIC_REWARD_COEFF,
        anneal_lr=ANNEAL_LR, tile_size=TILE_SIZE,
        gamma_int=GAMMA_INT, value_heads=2,
    )
    with open(result_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)

    np.savez_compressed(
        result_dir / "metrics.npz",
        episode_returns=np.array(ep_returns, dtype=np.float32),
        episode_lengths=np.array(ep_lengths, dtype=np.int32),
        episode_steps=np.array(ep_steps, dtype=np.int64),
        intrinsic_rewards=np.array(ir_means, dtype=np.float32),
    )

    torch.save(agent.state_dict(), result_dir / "model.pt")

    summary = dict(
        method=_full_name, env_id=env_id, seed=seed,
        final_return_100=float(np.mean(ep_returns[-100:])) if ep_returns else 0.0,
        final_length_100=float(np.mean(ep_lengths[-100:])) if ep_lengths else 0.0,
        total_episodes=len(ep_returns),
        wall_time_s=round(time.time() - start_time, 2),
        total_steps=global_step,
    )
    with open(result_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    # Bundle into a zip
    zip_path = Path(RESULTS_DIR) / f"{run_name}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fp in sorted(result_dir.iterdir()):
            zf.write(fp, f"{run_name}/{fp.name}")

    if USE_WANDB and WANDB_AVAILABLE:
        wandb.finish(quiet=True)
    envs.close()
    print(f"  -> {result_dir}/  &  {zip_path}")
    return summary

In [ ]:
# Run All Experiments
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device:       {device}")
_full_name = f"{METHOD_NAME}{SUFFIX}"
print(f"Method:       {_full_name}")
print(f"Environments: {len(ENVIRONMENTS)}")
print(f"Seeds:        {SEEDS}")
print(f"Total runs:   {len(ENVIRONMENTS) * len(SEEDS)}")
print("=" * 60)

all_summaries = {}
for env_id, total_steps in ENVIRONMENTS.items():
    for seed in SEEDS:
        rn = f"{_full_name}__{env_id}__{seed}"
        print(f"\n{'─'*60}")
        print(f"  {env_id}  seed={seed}  steps={total_steps:,}")
        print(f"{'─'*60}")
        try:
            s = train(env_id, total_steps, seed, device)
            all_summaries[rn] = s
        except Exception as exc:
            print(f"  FAILED: {exc}")
            import traceback; traceback.print_exc()
            all_summaries[rn] = {"error": str(exc)}

print("\n" + "=" * 72)
print(f"{'Run':<52} {'Return':>10} {'Time':>8}")
print("=" * 72)
for name, s in all_summaries.items():
    if "error" in s:
        print(f"{name:<52} {'ERROR':>10}")
    else:
        print(f"{name:<52} {s['final_return_100']:>10.3f} "
              f"{s['wall_time_s']:>7.1f}s")
print("=" * 72)
print("Done.")